# MiniTorch: Deep Learning Framework Demo

This notebook demonstrates the key features and capabilities of MiniTorch, a minimal deep learning framework built from scratch. MiniTorch implements core components including automatic differentiation, tensor operations, CUDA acceleration, and a complete decoder-only transformer architecture.

## Table of Contents

1. [Setup and Installation](#setup)
2. [Basic Tensor Operations](#tensors)
3. [Neural Network Functions](#nn-functions)
4. [Basic Neural Network Modules](#basic-modules)
5. [Transformer Architecture](#transformer)
6. [Sentiment Analysis Demo](#sentiment)
7. [Machine Translation Demo](#translation)

## 1. Setup and Installation <a name="setup"></a>

First, let's install the required dependencies and import necessary modules.

In [ ]:
# Install dependencies (uncomment if running for the first time)
# !pip install -r requirements.extra.txt
# !pip install -r requirements.txt
# !pip install -e .

# Compile CUDA kernels if GPU is available (uncomment if needed)
# !bash compile_cuda.sh

In [ ]:
# Import necessary modules
import numpy as np
import minitorch
from minitorch import (
    Tensor, 
    tensor, 
    rand,
    zeros,
    ones,
)
from minitorch.modules_basic import Linear, Dropout, LayerNorm1d, Embedding
from minitorch.modules_transfomer import MultiHeadAttention, TransformerLayer, DecoderLM
from minitorch.nn import softmax, softmax_loss, logsumexp

print("MiniTorch imported successfully!")

# Check CUDA availability
try:
    import numba
    if numba.cuda.is_available():
        print("CUDA is available!")
        from minitorch.cuda_kernel_ops import CudaKernelOps
        backend = CudaKernelOps()
    else:
        print("CUDA not available, using CPU backend")
        backend = None
except:
    print("Using CPU backend")
    backend = None

## 2. Basic Tensor Operations <a name="tensors"></a>

MiniTorch provides a custom tensor implementation with automatic differentiation support. Let's explore basic tensor operations.

In [ ]:
# Create tensors
a = tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
b = tensor([[7.0, 8.0, 9.0], [10.0, 11.0, 12.0]])

print("Tensor a:")
print(a)
print(f"Shape: {a.shape}\n")

print("Tensor b:")
print(b)
print(f"Shape: {b.shape}\n")

# Element-wise operations
c = a + b
print("a + b:")
print(c)

d = a * b
print("\na * b (element-wise):")
print(d)

# Matrix operations
e = tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
f = a @ e.permute(1, 0)  # Matrix multiplication
print("\nMatrix multiplication a @ e.T:")
print(f)

In [ ]:
# Reduction operations
print("Sum along dimension 1:")
print(a.sum(1))

print("\nMean along dimension 0:")
print(a.mean(0))

## 3. Neural Network Functions <a name="nn-functions"></a>

MiniTorch implements key neural network functions including softmax, logsumexp, and softmax loss.

In [ ]:
# Logsumexp - numerically stable log-sum-exp
logits = tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print("Input logits:")
print(logits)

lse = logsumexp(logits, dim=1)
print("\nLogsumexp along dimension 1:")
print(lse)

In [ ]:
# Softmax
probs = softmax(logits, dim=1)
print("Softmax probabilities (should sum to 1 along dim 1):")
print(probs)
print("\nSum along dimension 1:")
print(probs.sum(1))

In [ ]:
# Softmax loss (cross-entropy)
# logits shape: (batch_size, num_classes)
# targets shape: (batch_size,)
batch_logits = tensor([[2.0, 1.0, 0.1], [0.5, 2.5, 1.0]])
targets = tensor([0.0, 1.0])  # True class indices

loss = softmax_loss(batch_logits, targets)
print("Softmax loss (cross-entropy):")
print(loss)
print("\nMean loss:")
print(loss.sum() / loss.shape[0])

## 4. Basic Neural Network Modules <a name="basic-modules"></a>

MiniTorch provides modular building blocks for neural networks.

In [ ]:
# Linear layer
linear = Linear(in_features=10, out_features=5, backend=backend)
x = rand((3, 10), backend=backend)  # Batch of 3 samples

print("Input shape:", x.shape)
output = linear.forward(x)
print("Output shape after Linear(10, 5):", output.shape)
print("Output:")
print(output)

In [ ]:
# Dropout layer
dropout = Dropout(p_dropout=0.5)
x = ones((4, 8), backend=backend)

print("Original tensor (all ones):")
print(x)

# Set to training mode
dropout.train()
output_train = dropout.forward(x)
print("\nAfter dropout (training mode, ~50% zeros):")
print(output_train)

# Set to evaluation mode
dropout.eval()
output_eval = dropout.forward(x)
print("\nAfter dropout (eval mode, no dropout):")
print(output_eval)

In [ ]:
# Layer Normalization
layer_norm = LayerNorm1d(dim=8, backend=backend)
x = rand((4, 8), backend=backend)

print("Input:")
print(x)
print("\nMean along features:", x.mean(1))

output = layer_norm.forward(x)
print("\nAfter LayerNorm:")
print(output)
print("\nMean along features (should be ~0):", output.mean(1))

In [ ]:
# Embedding layer
vocab_size = 1000
embedding_dim = 64
embedding = Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim, backend=backend)

# Input: word indices
word_indices = tensor([[1.0, 5.0, 10.0], [2.0, 8.0, 15.0]])
print("Word indices shape:", word_indices.shape)

embedded = embedding.forward(word_indices)
print("\nEmbedded output shape:", embedded.shape)
print("Expected shape: (batch_size=2, seq_len=3, embedding_dim=64)")

## 5. Transformer Architecture <a name="transformer"></a>

MiniTorch implements a complete decoder-only transformer architecture based on GPT-2.

In [ ]:
# Multi-Head Attention
n_embd = 128
n_head = 4
batch_size = 2
seq_len = 8

mha = MultiHeadAttention(n_embd=n_embd, n_head=n_head, causal=True, backend=backend)
x = rand((batch_size, seq_len, n_embd), backend=backend)

print("Input shape:", x.shape)
output = mha.forward(x)
print("Output shape after MultiHeadAttention:", output.shape)
print("Input and output shapes match:", x.shape == output.shape)

In [ ]:
# Transformer Layer (with pre-LN architecture)
transformer_layer = TransformerLayer(n_embd=n_embd, n_head=n_head, backend=backend)
x = rand((batch_size, seq_len, n_embd), backend=backend)

print("Input shape:", x.shape)
output = transformer_layer.forward(x)
print("Output shape after TransformerLayer:", output.shape)

In [ ]:
# Complete Decoder Language Model
vocab_size = 10000
n_embd = 256
n_head = 8
n_layers = 4
max_len = 512

model = DecoderLM(
    n_vocab=vocab_size,
    n_embd=n_embd,
    n_head=n_head,
    n_layers=n_layers,
    max_len=max_len,
    backend=backend
)

print(f"Model created with:")
print(f"  Vocabulary size: {vocab_size}")
print(f"  Embedding dimension: {n_embd}")
print(f"  Number of heads: {n_head}")
print(f"  Number of layers: {n_layers}")
print(f"  Max sequence length: {max_len}")

# Forward pass
batch_size = 2
seq_len = 16
input_ids = rand((batch_size, seq_len), backend=backend) * vocab_size  # Random word indices
input_ids = input_ids.contiguous()

print(f"\nInput shape: {input_ids.shape}")
logits = model.forward(input_ids)
print(f"Output logits shape: {logits.shape}")
print(f"Expected: (batch_size={batch_size}, seq_len={seq_len}, vocab_size={vocab_size})")

## 6. Sentiment Analysis Demo <a name="sentiment"></a>

MiniTorch includes a sentiment analysis pipeline. Here's how to use it:

```bash
python project/run_sentiment_linear.py
```

The sentiment analysis pipeline:
- Uses a simple linear model for text classification
- Trains on sentiment labeled data
- Evaluates model performance on test data

**Note**: This requires the full training pipeline and is best run from the command line.

## 7. Machine Translation Demo <a name="translation"></a>

MiniTorch implements a complete machine translation pipeline using the transformer architecture.

### Running Machine Translation

To train a German-to-English translation model:

```bash
python project/run_machine_translation.py
```

### Pipeline Overview

The machine translation pipeline includes:
1. **Dataset Loading**: Automatically downloads and preprocesses IWSLT14 German-English dataset
2. **Tokenization**: Uses byte-level BPE tokenizer
3. **Model Training**: Trains decoder-only transformer with:
   - Vocabulary size: 10,000
   - Embedding dimension: 256
   - Learning rate: 0.02
   - Multiple epochs with validation
4. **Evaluation**: Computes BLEU scores on test set
5. **Generation**: Argmax decoding for translation

### Expected Performance

- **First epoch**: BLEU score ~7
- **After 10 epochs**: BLEU score ~20
- **Training time**: ~1 hour per epoch on A10G GPU
- **Step time**: ~25 seconds per training step

### Output

Results are saved in `./workdir_vocab10000_lr0.02_embd256/` including:
- Model checkpoints
- Training logs
- Generated translations
- BLEU scores

**Note**: Full training requires significant compute resources and is best run on a GPU-equipped machine.

## Summary

This notebook demonstrated the key features of MiniTorch:

1. **Tensor Operations**: Custom tensor implementation with automatic differentiation
2. **Neural Network Functions**: Softmax, logsumexp, cross-entropy loss
3. **Basic Modules**: Linear layers, dropout, layer normalization, embeddings
4. **Transformer Architecture**: Multi-head attention, transformer layers, complete decoder model
5. **NLP Applications**: Sentiment analysis and machine translation pipelines

MiniTorch provides a complete, educational deep learning framework built from scratch, demonstrating how modern neural network architectures work under the hood.

### Next Steps

- Explore the source code in `minitorch/` to understand implementation details
- Run the full training pipelines for sentiment analysis and machine translation
- Experiment with different model architectures and hyperparameters
- Try CUDA acceleration for faster training on GPU

### Testing

Run comprehensive tests:
```bash
# Test all components
python -m pytest -l -v

# Test specific modules
python -m pytest -l -v -k "test_linear_student"
python -m pytest -l -v -k "test_transformer"
```